<a href="https://colab.research.google.com/github/darcor99/Options_Volatility_plots/blob/main/Options_volatility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install alpaca-py
!pip install alpaca-trade-api
!pip install plotly
!pip install tensorflow
!pip install scikit-learn
!pip install yfinance

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import GetAssetsRequest
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy.stats import norm
import yfinance as yf

In [3]:
# Set all seeds
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
!curl -v https://paper-api.alpaca.markets/v2/account

In [5]:
from google.colab import userdata
API_KEY = userdata.get('alpaca_api_key')
SECRET_KEY = userdata.get('alpaca_secret_key')

##setup trading client
trade_client = TradingClient(api_key=API_KEY, secret_key=SECRET_KEY, paper=True, url_override=None)
#trade_client = TradingClient(api_key=API_KEY, secret_key=SECRET_KEY, paper=False, url_override=None)

In [ ]:
# check trading account
# You can check definition of each field in the following documents
# ref. https://docs.alpaca.markets/docs/account-plans
# ref. https://docs.alpaca.markets/reference/getaccount-1
acct = trade_client.get_account()
acct

In [7]:
##historical data client
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)
client

In [29]:
#######################
symbol = "BITF" ###you can change this with any US company's ticker

In [30]:
from datetime import datetime, timedelta
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# Define time range
end_date = datetime.now() - timedelta(hours=1) # Exclude last 30 days
start_date = end_date - timedelta(days=365)  # last 1 year before the last 30 days

request_params = StockBarsRequest(
    symbol_or_symbols=symbol,
    timeframe=TimeFrame.Day,
    start=start_date,
    end=end_date,
)

bars = client.get_stock_bars(request_params)

# Access the data for the specific symbol using the multi-index
df = bars.df.loc[symbol]
df.head()

,open,high,low,close,volume,trade_count,vwap
timestamp,,,,,,,
2025-01-15 05:00:00+00:00,1.64,1.69,1.61,1.67,45215165.0,25891.0,1.661415
2025-01-16 05:00:00+00:00,1.65,1.69,1.62,1.64,38016910.0,25250.0,1.656894
2025-01-17 05:00:00+00:00,1.72,1.78,1.67,1.68,49313603.0,28153.0,1.730053
2025-01-21 05:00:00+00:00,1.74,1.74,1.63,1.67,53222718.0,31500.0,1.677659
2025-01-22 05:00:00+00:00,1.62,1.68,1.58,1.66,46341318.0,31074.0,1.623453


In [31]:
def plot_stock_data(df):
  ##calculate SMAs (Simple moving averages)
  df['SMA5'] = df['close'].rolling(window=5).mean()
  df['SMA20'] = df['close'].rolling(window=20).mean()
  df['SMA50'] = df['close'].rolling(window=50).mean()

  fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
  vertical_spacing=0.03,
  subplot_titles=(f'{symbol} candlesticks with SMAs', 'volume'),
  row_width=[0.2, 0.7])

  #add candlesticks
  fig.add_trace(go.Candlestick(x=df.index,
  open=df['open'],
  high=df['high'],
  low=df['low'],
  close=df['close'],
  name='candlesticks'),
  row=1, col=1)

  ##add SMAs
  fig.add_trace(go.Scatter(x=df.index,
  y=df['SMA5'],
  name='SMA5'),
  row=1, col=1)

  fig.add_trace(go.Scatter(x=df.index,
  y=df['SMA20'],
  name='SMA20'),
  row=1, col=1)

  fig.add_trace(go.Scatter(x=df.index,
  y=df['SMA50'],
  name='SMA50'),
  row=1, col=1)

  fig.update_layout(title=f'{symbol} Price and volume analysis',
  xaxis_rangeslider_visible=False,
  yaxis_title = 'price (USD)',
  yaxis2_title = 'volume',
  height = 800)

  fig.show()

In [32]:
plot_stock_data(df)

In [33]:
fig_year = go.Figure(data=[go.Candlestick(
    x=df.index,
    open=df['open'],
    high=df['high'],
    low=df['low'],
    close=df['close'],
    increasing_line_color='green',
    decreasing_line_color='red'
)])
fig_year.update_layout(
    title=f"{symbol} Daily Candlestick",
    xaxis_title="Date",
    yaxis_title="Price",
    xaxis_rangeslider_visible=True
)
fig_year.show()

In [34]:
# Fetch options data using Yahoo Finance
ticker = yf.Ticker(symbol)

# Get available expiration dates
expiration_dates = ticker.options
print(f"Available expiration dates for {symbol}:")
print(expiration_dates[:10])  # Show first 10 expiration dates

Available expiration dates for BITF:
('2026-01-16', '2026-01-23', '2026-01-30', '2026-02-06', '2026-02-13', '2026-02-20', '2026-02-27', '2026-03-20', '2026-05-15', '2026-06-18')


In [35]:
# Fetch options chain for multiple expiration dates
# Fetch ALL available expiration dates for a denser volatility surface

all_calls = []
all_puts = []

# Use all available expirations for a continuous surface
num_expirations = len(expiration_dates)
print(f"Fetching options for {num_expirations} expiration dates...")

for i, exp_date in enumerate(expiration_dates):
    try:
        opt_chain = ticker.option_chain(exp_date)

        # Add expiration date column
        calls = opt_chain.calls.copy()
        calls['expiration_date'] = exp_date
        calls['option_type'] = 'call'

        puts = opt_chain.puts.copy()
        puts['expiration_date'] = exp_date
        puts['option_type'] = 'put'

        all_calls.append(calls)
        all_puts.append(puts)

        if (i + 1) % 10 == 0 or i == num_expirations - 1:
            print(f"Progress: {i + 1}/{num_expirations} expirations fetched")
    except Exception as e:
        print(f"Error fetching {exp_date}: {e}")

# Combine all data
calls_df = pd.concat(all_calls, ignore_index=True)
puts_df = pd.concat(all_puts, ignore_index=True)

print(f"\nTotal calls: {len(calls_df)}, Total puts: {len(puts_df)}")
print(f"Expiration range: {expiration_dates[0]} to {expiration_dates[-1]}")

Fetching options for 14 expiration dates...
Progress: 10/14 expirations fetched
Progress: 14/14 expirations fetched

Total calls: 175, Total puts: 149
Expiration range: 2026-01-16 to 2028-01-21


In [36]:
# Examine the structure of Yahoo Finance options data
print("Columns available in options data:")
print(calls_df.columns.tolist())
print("\nSample call options data:")
calls_df.head()

Columns available in options data:
['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask', 'change', 'percentChange', 'volume', 'openInterest', 'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'expiration_date', 'option_type']

Sample call options data:


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration_date,option_type
0,BITF260116C00000500,2026-01-14 20:01:22+00:00,0.5,2.40,2.12,2.76,-0.24,-9.090909,56.0,2567,0.500005,True,REGULAR,USD,2026-01-16,call
1,BITF260116C00001000,2026-01-14 20:52:02+00:00,1.0,1.93,1.73,2.04,-0.17,-8.095237,195.0,12369,10.562503,True,REGULAR,USD,2026-01-16,call
2,BITF260116C00001500,2026-01-14 20:44:39+00:00,1.5,1.45,1.38,1.51,-0.20,-12.121207,114.0,11378,3.500001,True,REGULAR,USD,2026-01-16,call
3,BITF260116C00002000,2026-01-14 20:49:21+00:00,2.0,0.93,0.91,1.00,-0.22,-19.130432,403.0,19548,2.750003,True,REGULAR,USD,2026-01-16,call
4,BITF260116C00002500,2026-01-14 20:32:48+00:00,2.5,0.45,0.41,0.48,-0.18,-28.571428,3180.0,20670,1.062505,True,REGULAR,USD,2026-01-16,call


In [37]:
# Create a standardized options DataFrame for analysis
# Yahoo Finance provides: contractSymbol, strike, lastPrice, bid, ask, impliedVolatility, etc.

options_df = pd.concat([calls_df, puts_df], ignore_index=True)

# Rename columns to match the expected format
options_df = options_df.rename(columns={
    'contractSymbol': 'symbol',
    'strike': 'strike_price',
    'lastPrice': 'last_price',
    'bid': 'bid_price',
    'ask': 'ask_price',
    'impliedVolatility': 'yf_implied_volatility',  # Yahoo's IV for comparison
    'volume': 'volume',
    'openInterest': 'open_interest'
})

# Add underlying symbol
options_df['underlying'] = symbol

print(f"Total options: {len(options_df)}")
options_df.head()

Total options: 324


,symbol,lastTradeDate,strike_price,last_price,bid_price,ask_price,change,percentChange,volume,open_interest,yf_implied_volatility,inTheMoney,contractSize,currency,expiration_date,option_type,underlying
0,BITF260116C00000500,2026-01-14 20:01:22+00:00,0.5,2.40,2.12,2.76,-0.24,-9.090909,56.0,2567,0.500005,True,REGULAR,USD,2026-01-16,call,BITF
1,BITF260116C00001000,2026-01-14 20:52:02+00:00,1.0,1.93,1.73,2.04,-0.17,-8.095237,195.0,12369,10.562503,True,REGULAR,USD,2026-01-16,call,BITF
2,BITF260116C00001500,2026-01-14 20:44:39+00:00,1.5,1.45,1.38,1.51,-0.20,-12.121207,114.0,11378,3.500001,True,REGULAR,USD,2026-01-16,call,BITF
3,BITF260116C00002000,2026-01-14 20:49:21+00:00,2.0,0.93,0.91,1.00,-0.22,-19.130432,403.0,19548,2.750003,True,REGULAR,USD,2026-01-16,call,BITF
4,BITF260116C00002500,2026-01-14 20:32:48+00:00,2.5,0.45,0.41,0.48,-0.18,-28.571428,3180.0,20670,1.062505,True,REGULAR,USD,2026-01-16,call,BITF


In [38]:
df['returns'] = df['close'].pct_change()
sigma = df['returns'].std() * np.sqrt(252)  # annualized
sigma

np.float64(1.046462857099832)

In [39]:
print(options_df.shape[0])

324


# Task
To analyze the implied volatility for call options of AMD, I will filter the options data for a specific expiration date (2025-12-05) and option type (call). Then, I'll extract the current stock price from the historical data and compute the time to expiration. Finally, I will define a Black-Scholes function to calculate the implied volatility for each option and visualize the results by plotting implied volatility against strike prices to observe any volatility smile or smirk.

## Filter Options Data

### Subtask:
Filter the `options_df` DataFrame to select call options for a specific expiration date (2025-12-05). Extract the current underlying stock price from the `df` DataFrame and calculate the time to expiration for the selected options.


**Reasoning**:
The subtask requires filtering the options data, extracting the current stock price, and calculating the time to expiration. I will perform these steps sequentially within a single code block to complete the initial data preparation for option pricing.



## Filter Options Data

### Subtask:
Filter the `options_df` DataFrame to select call options for a specific expiration date (2025-12-05). Extract the current underlying stock price from the `df` DataFrame and calculate the time to expiration for the selected options.


**Reasoning**:
I will filter the options data, extract the current stock price, and calculate the time to expiration as per the instructions. This involves setting an expiration date filter, filtering the dataframe, getting the last stock price, converting dates to datetime objects, and calculating the time difference.



In [40]:
# Get current stock price
S = df['close'].iloc[-1]

# Convert expiration_date to datetime
options_df['expiration_date'] = pd.to_datetime(options_df['expiration_date'])

# Get the current date from the last entry in df's index
current_date = pd.to_datetime(df.index[-1].date())

# Calculate time to expiration in years
options_df['T'] = (options_df['expiration_date'] - current_date).dt.days / 365.0

# Filter out options with T <= 0 (already expired)
options_df = options_df[options_df['T'] > 0].copy()

print(f"Current Stock Price (S): ${S:.2f}")
print(f"Current Date: {current_date.date()}")
print(f"Options with valid T: {len(options_df)}")
print(f"\nTime to Expiration range: {options_df['T'].min():.4f} to {options_df['T'].max():.4f} years")
options_df.head()

Current Stock Price (S): $2.94
Current Date: 2026-01-14
Options with valid T: 324

Time to Expiration range: 0.0055 to 2.0192 years


,symbol,lastTradeDate,strike_price,last_price,bid_price,ask_price,change,percentChange,volume,open_interest,yf_implied_volatility,inTheMoney,contractSize,currency,expiration_date,option_type,underlying,T
0,BITF260116C00000500,2026-01-14 20:01:22+00:00,0.5,2.40,2.12,2.76,-0.24,-9.090909,56.0,2567,0.500005,True,REGULAR,USD,2026-01-16,call,BITF,0.005479
1,BITF260116C00001000,2026-01-14 20:52:02+00:00,1.0,1.93,1.73,2.04,-0.17,-8.095237,195.0,12369,10.562503,True,REGULAR,USD,2026-01-16,call,BITF,0.005479
2,BITF260116C00001500,2026-01-14 20:44:39+00:00,1.5,1.45,1.38,1.51,-0.20,-12.121207,114.0,11378,3.500001,True,REGULAR,USD,2026-01-16,call,BITF,0.005479
3,BITF260116C00002000,2026-01-14 20:49:21+00:00,2.0,0.93,0.91,1.00,-0.22,-19.130432,403.0,19548,2.750003,True,REGULAR,USD,2026-01-16,call,BITF,0.005479
4,BITF260116C00002500,2026-01-14 20:32:48+00:00,2.5,0.45,0.41,0.48,-0.18,-28.571428,3180.0,20670,1.062505,True,REGULAR,USD,2026-01-16,call,BITF,0.005479


## Calculate Implied Volatility

### Subtask:
Define a function to calculate implied volatility for each option using the Black-Scholes model, incorporating the filtered options data, current stock price, strike prices, time to expiration, and a risk-free rate. This will likely involve a numerical solver.


**Reasoning**:
I will define the Black-Scholes formula for call options and a function to calculate implied volatility using `scipy.optimize.brentq`. Then, I'll apply these functions to the `filtered_options_df` to compute and store the implied volatility for each option, using a risk-free rate of 0.05 and handling potential missing bid/ask prices by falling back to the last traded price.



In [41]:
from scipy.stats import norm
from scipy.optimize import brentq

# Black-Scholes formula for a call option
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

# Black-Scholes formula for a put option
def black_scholes_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return put_price

# Function to calculate implied volatility
def implied_volatility(market_price, S, K, T, r, option_type='call'):
    if market_price <= 0 or T <= 0:
        return np.nan

    # Choose the appropriate pricing function
    if option_type == 'call':
        pricing_func = black_scholes_call
    else:
        pricing_func = black_scholes_put

    def objective_function(sigma):
        return pricing_func(S, K, T, r, sigma) - market_price

    try:
        implied_vol = brentq(objective_function, 1e-6, 5.0)
        return implied_vol
    except ValueError:
        return np.nan

# Set risk-free rate
r = 0.05

# Apply the implied volatility calculation
implied_vols = []
for index, row in options_df.iterrows():
    # Use mid price if bid/ask available, otherwise last price
    if row['bid_price'] > 0 and row['ask_price'] > 0:
        market_price = (row['bid_price'] + row['ask_price']) / 2
    else:
        market_price = row['last_price']

    if pd.isna(market_price) or market_price <= 0:
        implied_vols.append(np.nan)
        continue

    iv = implied_volatility(market_price, S, row['strike_price'], row['T'], r, row['option_type'])
    implied_vols.append(iv)

options_df['implied_volatility'] = implied_vols

# Compare our calculated IV with Yahoo's IV
valid_iv = options_df.dropna(subset=['implied_volatility', 'yf_implied_volatility'])
print(f"Successfully calculated IV for {len(valid_iv)} options")
print(f"\nComparison of calculated IV vs Yahoo Finance IV (first 10):")
print(valid_iv[['symbol', 'strike_price', 'option_type', 'implied_volatility', 'yf_implied_volatility']].head(10))

Successfully calculated IV for 267 options

Comparison of calculated IV vs Yahoo Finance IV (first 10):
                 symbol  strike_price option_type  implied_volatility  \
2   BITF260116C00001500           1.5        call            4.371367   
3   BITF260116C00002000           2.0        call            3.302210   
4   BITF260116C00002500           2.5        call            1.267817   
5   BITF260116C00003000           3.0        call            1.337264   
6   BITF260116C00003500           3.5        call            1.690873   
7   BITF260116C00004000           4.0        call            2.387373   
8   BITF260116C00004500           4.5        call            3.082034   
9   BITF260116C00005000           5.0        call            3.674214   
10  BITF260116C00005500           5.5        call            4.191390   
11  BITF260116C00006000           6.0        call            4.650692   

    yf_implied_volatility  
2                3.500001  
3                2.750003  
4       

In [42]:
print(options_df.tail())

                  symbol             lastTradeDate  strike_price  last_price  \
319  BITF280121P00005000 2026-01-08 15:32:35+00:00           5.0        2.90   
320  BITF280121P00005500 2025-10-21 13:56:07+00:00           5.5        3.31   
321  BITF280121P00007000 2025-11-11 20:55:34+00:00           7.0        5.06   
322  BITF280121P00010000 2025-11-10 19:41:35+00:00          10.0        7.30   
323  BITF280121P00012000 2026-01-13 20:21:28+00:00          12.0        9.17   

     bid_price  ask_price  change  percentChange  volume  open_interest  \
319       2.29       3.40     0.0            0.0     1.0            102   
320       2.26       4.95     0.0            0.0    40.0            200   
321       0.00       0.00     0.0            0.0     1.0             79   
322       0.00       0.00     0.0            0.0     8.0             97   
323       7.00      11.30     0.0            0.0    12.0            102   

     yf_implied_volatility  inTheMoney contractSize currency expirat

## Plot Implied Volatility

### Subtask:
Generate a plot of implied volatility against the strike prices to visualize the volatility smile or smirk. The plot will include appropriate labels and a legend.


**Reasoning**:
I will filter out rows with NaN values in 'implied_volatility' to ensure data integrity for plotting. Then, I will create a scatter plot using Plotly to visualize the relationship between implied volatility and strike price, adding appropriate labels and a title to enhance readability and meet the subtask requirements.



In [43]:
import plotly.express as px

# Filter out NaN values from 'implied_volatility'
plot_df = options_df.dropna(subset=['implied_volatility'])

# Filter for reasonable IV values (exclude extreme outliers)
plot_df = plot_df[(plot_df['implied_volatility'] > 0) & (plot_df['implied_volatility'] < 3)]

# Create the scatter plot with color by option type
fig = px.scatter(
    plot_df,
    x='strike_price',
    y='implied_volatility',
    color='option_type',
    symbol='expiration_date',
    title=f'Implied Volatility vs. Strike Price for {symbol} Options (Yahoo Finance)',
    labels={
        'strike_price': 'Strike Price ($)',
        'implied_volatility': 'Implied Volatility',
        'option_type': 'Option Type',
        'expiration_date': 'Expiration'
    },
    hover_data=['symbol', 'last_price', 'bid_price', 'ask_price']
)

# Add vertical line for current stock price
fig.add_vline(x=S, line_dash="dash", line_color="gray",
              annotation_text=f"Current Price: ${S:.2f}")

fig.update_layout(height=600)
fig.show()

In [44]:
import plotly.graph_objects as go
from scipy.interpolate import griddata

# Filter for calls only to show volatility surface
plot_df_3d = options_df[options_df['option_type'] == 'call'].dropna(subset=['implied_volatility', 'T', 'strike_price'])

# Filter for reasonable IV values
plot_df_3d = plot_df_3d[(plot_df_3d['implied_volatility'] > 0) & (plot_df_3d['implied_volatility'] < 3)]

# Extract data points
strikes = plot_df_3d['strike_price'].values
times = plot_df_3d['T'].values
ivs = plot_df_3d['implied_volatility'].values

# Create a regular grid for interpolation
strike_range = np.linspace(strikes.min(), strikes.max(), 50)
time_range = np.linspace(times.min(), times.max(), 50)
strike_grid, time_grid = np.meshgrid(strike_range, time_range)

# Interpolate IV values onto the grid using cubic interpolation
iv_grid = griddata((strikes, times), ivs, (strike_grid, time_grid), method='cubic')

# Fill NaN values with linear interpolation as fallback
iv_grid_linear = griddata((strikes, times), ivs, (strike_grid, time_grid), method='linear')
iv_grid = np.where(np.isnan(iv_grid), iv_grid_linear, iv_grid)

# Create the 3D surface plot
fig_3d = go.Figure()

# Add the interpolated surface
fig_3d.add_trace(go.Surface(
    x=strike_grid,
    y=time_grid,
    z=iv_grid,
    colorscale='Viridis',
    opacity=0.9,
    colorbar=dict(title='IV'),
    name='IV Surface',
    hovertemplate='Strike: $%{x:.0f}<br>T: %{y:.4f} years<br>IV: %{z:.2%}<extra></extra>'
))

# Add scatter points for actual data
fig_3d.add_trace(go.Scatter3d(
    x=strikes,
    y=times,
    z=ivs,
    mode='markers',
    marker=dict(size=3, color='red', opacity=0.6),
    name='Data Points',
    hovertemplate='Strike: $%{x:.0f}<br>T: %{y:.4f} years<br>IV: %{z:.2%}<extra></extra>'
))

# Update layout
fig_3d.update_layout(
    title=f'3D Volatility Surface for {symbol} Call Options (Interpolated)',
    scene=dict(
        xaxis_title='Strike Price ($)',
        yaxis_title='Time to Expiry (Years)',
        zaxis_title='Implied Volatility'
    ),
    height=700
)

fig_3d.show()

print(f"Surface interpolated from {len(plot_df_3d)} data points")
print(f"Strike range: ${strikes.min():.0f} - ${strikes.max():.0f}")
print(f"Time range: {times.min():.4f} - {times.max():.2f} years")

Surface interpolated from 142 data points
Strike range: $0 - $12
Time range: 0.0055 - 2.02 years


In [45]:
# 3D Surface Plot: Moneyness vs Time to Expiry vs Implied Volatility

# Calculate moneyness (S/K ratio)
# Moneyness > 1 means ITM for calls, OTM for puts
# Moneyness < 1 means OTM for calls, ITM for puts
# Moneyness = 1 means ATM

options_df['moneyness'] = S / options_df['strike_price']

# Filter for calls to show volatility surface
plot_df_moneyness = options_df[options_df['option_type'] == 'call'].dropna(subset=['implied_volatility', 'T', 'moneyness'])

# Filter for reasonable IV values and moneyness range
plot_df_moneyness = plot_df_moneyness[
    (plot_df_moneyness['implied_volatility'] > 0) &
    (plot_df_moneyness['implied_volatility'] < 3) &
    (plot_df_moneyness['moneyness'] > 0.5) &
    (plot_df_moneyness['moneyness'] < 1.5)
]

# Extract data points
moneyness_vals = plot_df_moneyness['moneyness'].values
times_m = plot_df_moneyness['T'].values
ivs_m = plot_df_moneyness['implied_volatility'].values

# Create a regular grid for interpolation
moneyness_range = np.linspace(moneyness_vals.min(), moneyness_vals.max(), 50)
time_range_m = np.linspace(times_m.min(), times_m.max(), 50)
moneyness_grid, time_grid_m = np.meshgrid(moneyness_range, time_range_m)

# Interpolate IV values onto the grid using cubic interpolation
iv_grid_m = griddata((moneyness_vals, times_m), ivs_m, (moneyness_grid, time_grid_m), method='cubic')

# Fill NaN values with linear interpolation as fallback
iv_grid_m_linear = griddata((moneyness_vals, times_m), ivs_m, (moneyness_grid, time_grid_m), method='linear')
iv_grid_m = np.where(np.isnan(iv_grid_m), iv_grid_m_linear, iv_grid_m)

# Create the 3D surface plot
fig_moneyness = go.Figure()

# Add the interpolated surface
fig_moneyness.add_trace(go.Surface(
    x=moneyness_grid,
    y=time_grid_m,
    z=iv_grid_m,
    colorscale='Viridis',
    opacity=0.9,
    colorbar=dict(title='IV'),
    name='IV Surface',
    hovertemplate='Moneyness: %{x:.3f}<br>T: %{y:.4f} years<br>IV: %{z:.2%}<extra></extra>'
))

# Add scatter points for actual data
fig_moneyness.add_trace(go.Scatter3d(
    x=moneyness_vals,
    y=times_m,
    z=ivs_m,
    mode='markers',
    marker=dict(size=3, color='red', opacity=0.5),
    name='Data Points'
))

# Add vertical line at ATM (moneyness = 1)
atm_times = np.linspace(times_m.min(), times_m.max(), 20)
atm_ivs = griddata((moneyness_vals, times_m), ivs_m, (np.ones_like(atm_times), atm_times), method='linear')
fig_moneyness.add_trace(go.Scatter3d(
    x=np.ones_like(atm_times),
    y=atm_times,
    z=atm_ivs,
    mode='lines',
    line=dict(color='red', width=5),
    name='ATM (Moneyness=1)'
))

# Update layout
fig_moneyness.update_layout(
    title=f'Volatility Surface: Moneyness vs Time to Expiry for {symbol} Call Options',
    scene=dict(
        xaxis_title='Moneyness (S/K)',
        yaxis_title='Time to Expiry (Years)',
        zaxis_title='Implied Volatility',
        xaxis=dict(
            tickvals=[0.6, 0.8, 1.0, 1.2, 1.4],
            ticktext=['0.6 (OTM)', '0.8', '1.0 (ATM)', '1.2', '1.4 (ITM)']
        )
    ),
    height=700,
    legend=dict(x=0.02, y=0.98)
)

fig_moneyness.show()

print(f"\nMoneyness Statistics:")
print(f"  Data points: {len(plot_df_moneyness)}")
print(f"  Moneyness range: {moneyness_vals.min():.3f} to {moneyness_vals.max():.3f}")
print(f"  Time range: {times_m.min():.4f} to {times_m.max():.2f} years")
print(f"  ATM options (0.95-1.05): {len(plot_df_moneyness[(plot_df_moneyness['moneyness'] >= 0.95) & (plot_df_moneyness['moneyness'] <= 1.05)])}")


Moneyness Statistics:
  Data points: 97
  Moneyness range: 0.535 to 1.470
  Time range: 0.0055 to 2.02 years
  ATM options (0.95-1.05): 14


## Final Task

### Subtask:
Summarize the implied volatility findings and discuss any observed patterns, such as the volatility smile or smirk, for the selected expiration date.


## Summary

### Data Source
Options data is now fetched from **Yahoo Finance** instead of Alpaca, providing more comprehensive options chain data without subscription limitations.

### Key Features
* Fetches options data for **ALL available expiration dates** for a dense volatility surface
* Includes both call and put options for analysis
* Yahoo Finance provides its own implied volatility calculation for comparison
* Black-Scholes model implemented for both calls and puts

### Visualization Improvements
* **Interpolated 3D surfaces** using `scipy.interpolate.griddata` with cubic interpolation
* Smooth, continuous volatility surfaces instead of sparse scatter plots
* 50x50 grid resolution for high-quality surface rendering
* Actual data points overlaid in red for reference
* ATM line highlighted on moneyness plot

### Analysis Results
* The volatility smile/smirk pattern can be observed in the strike price vs IV plot
* Multiple expiration dates allow visualization of the full volatility surface
* Our calculated IV can be compared against Yahoo Finance's provided IV values

### Observations
* Implied volatility tends to be higher for out-of-the-money options (both puts and calls)
* The volatility surface shows how IV varies across both strike prices and time to expiration
* Near-term options often exhibit more pronounced volatility smiles
* Term structure of volatility visible across different expiration dates

### Next Steps
* Compare IV patterns across different underlying assets
* Analyze historical volatility vs implied volatility spreads
* Add put options surface for comparison